In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests

BASE_URL = "http://books.toscrape.com/"
response = requests.get(BASE_URL)
soup = BeautifulSoup(response.content, "html.parser")

category_links = (
    soup.find("div", class_="side_categories")
    .find("ul")
    .find("ul")
    .find_all("a")
)
target_categories = ["Travel","Mystery","Historical Fiction","Sequential Art",]
all_books = []

for link in category_links:
  category_name = link.text.strip()
  if category_name in target_categories:
    category_url = BASE_URL + link["href"]
    category_response = requests.get(category_url)
    category_soup = BeautifulSoup(category_response.content, "html.parser")
    books = category_soup.find_all("article", class_="product_pod")
    for book in books:
      title = book.find("h3").find("a")["title"]
      price = book.find("p", class_="price_color").text.strip()
      rating = book.find("p", class_="star-rating")
      star_rating = rating["class"][1]
      availability = book.find("p", class_="instock availability").text.strip()
      all_books.append({
          "title": title,
          "price": price,
          "star_rating": star_rating,
          "availability": availability,
          "category": category_name,
      })
df_raw = pd.DataFrame(all_books)
print(f"{len(df_raw)} books scraped successfully")
display(df_raw.head())

71 books scraped successfully


,title,price,star_rating,availability,category
0,It's Only the Himalayas,£45.17,Two,In stock,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel


In [ ]:
import numpy as np
GBP_TO_INR = 105.50
Rating_map ={"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
df_clean = df_raw.copy()

df_clean["price"] = df_clean["price"].str.replace("£","", regex=False).str.strip()
df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce")
if df_clean["price"].isnull().any():
  df_clean["price"].fillna(df_clean["price"].median(), inplace=True)
df_clean["price_inr"] = df_clean["price"] * GBP_TO_INR

df_clean["rating"] = df_clean["star_rating"].map(Rating_map)
if df_clean["rating"].isnull().any():
  df_clean["rating"].fillna(df_clean["rating"].median(), inplace=True)
df_clean["rating"] = df_clean["rating"].astype(int)

df_clean["in_stock"] = (
    df_clean["availability"].str.lower().str.contains("in stock").astype(int)
)

print("Data Cleaning Complete!")
display(
    df_clean[
        ["title", "category", "price", "price_inr", "rating", "in_stock"]
    ].head()
)

Data Cleaning Complete!


,title,category,price,price_inr,rating,in_stock
0,It's Only the Himalayas,Travel,45.17,4765.435,2,1
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,5214.865,4,1
2,See America: A Celebration of Our National Par...,Travel,48.87,5155.785,3,1
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,3897.170,2,1
4,Under the Tuscan Sun,Travel,37.33,3938.315,3,1


In [ ]:
import sqlite3
conn = sqlite3.connect("zepto_catalog.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    in_stock INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
);
""")
conn.commit()

for cat_name in df_clean["category"].unique():
  cursor.execute(
      "INSERT OR IGNORE INTO categories (category_name) VALUES (?);",
      (cat_name,),
  )
conn.commit()
cursor.execute("SELECT category_name, category_id FROM categories;")
category_map = dict(cursor.fetchall())
for _, row in df_clean.iterrows():
  cat_id = category_map[row["category"]]
  cursor.execute(
      """
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?);
    """,
      (
          row["title"],
          row["price"],
          row["price_inr"],
          row["rating"],
          row["in_stock"],
          cat_id,
      ),
  )
conn.commit()
print("Database created & loaded successfully!")

Database created & loaded successfully!


In [ ]:
q1 = "SELECT book_id, title, price_gbp, rating, in_stock FROM books WHERE rating = 5 AND in_stock = 1;"
print("Query 1: Available 5-Star Books")
display(pd.read_sql_query(q1, conn).head(3))

q2 = "SELECT book_id, title, price_inr, rating FROM books ORDER BY price_inr DESC LIMIT 5;"
print("\nQuery 2: Top 5 Most Expensive Books")
display(pd.read_sql_query(q2, conn))

q3 = "SELECT DISTINCT category_name FROM categories;"
print("\nQuery 3: Distinct Categories")
display(pd.read_sql_query(q3, conn))

q4 = "SELECT book_id, title, price_inr, rating FROM books WHERE price_inr BETWEEN 2000 AND 4000 ORDER BY price_inr ASC;"
print("\nQuery 4: Books Priced ₹2000-₹4000")
display(pd.read_sql_query(q4, conn).head(3))

q5 = """
SELECT b.book_id, b.title, c.category_name, b.rating, b.price_gbp, b.price_inr
FROM books b
INNER JOIN categories c ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY c.category_name ASC, b.rating DESC;
"""
print("\nQuery 5: SQL JOIN (Highest Rated Per Category)")
display(pd.read_sql_query(q5, conn).head(3))

Query 1: Available 5-Star Books


,book_id,title,price_gbp,rating,in_stock
0,11,"1,000 Places to See Before You Die",26.08,5,1
1,20,A Time of Torment (Charlie Parker #14),48.35,5,1
2,29,What Happened on Beale Street (Secrets of the ...,25.37,5,1



Query 2: Top 5 Most Expensive Books


,book_id,title,price_inr,rating
0,26,Boar Island (Anna Pigeon #19),6275.140,3
1,8,A Year in Provence (Provence #1),6000.840,4
2,14,The Past Never Ends,5960.750,4
3,49,The Last Painting of Sara de Vos,5860.525,2
4,34,A Flight of Arrows (The Pathfinders #2),5858.415,5



Query 3: Distinct Categories


,category_name
0,Travel
1,Mystery
2,Historical Fiction
3,Sequential Art



Query 4: Books Priced ₹2000-₹4000


,book_id,title,price_inr,rating
0,59,"Pop Gun War, Volume 1: Gift",2001.335,1
1,54,This One Summer,2056.195,4
2,13,"In a Dark, Dark Wood",2070.965,1



Query 5: SQL JOIN (Highest Rated Per Category)


,book_id,title,category_name,rating,price_gbp,price_inr
0,34,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,36,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
2,45,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760


In [ ]:
df_readback_q1 = pd.read_sql(
    "SELECT book_id, title, price_gbp, rating FROM books WHERE rating = 5 AND in_stock = 1;",
    conn,
)
print("Readback 1: Query 1 via pd.read_sql (5-star in-stock books)")
display(df_readback_q1)

join_query = """
SELECT b.book_id, b.title, c.category_name, b.rating, b.price_gbp, b.price_inr
FROM books b
INNER JOIN categories c ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY c.category_name ASC, b.rating DESC;
"""
df_readback_q5 = pd.read_sql(join_query, conn)
print("\nReadback 2: Query 5 (JOIN) via pd.read_sql")
display(df_readback_q5.head())

df_books_mem = pd.read_sql("SELECT * FROM books;", conn)
df_cats_mem = pd.read_sql("SELECT * FROM categories;", conn)
df_merged = pd.merge(df_books_mem, df_cats_mem, on="category_id", how="inner")
df_merge_result = (
    df_merged[df_merged["rating"] >= 4][
        ["book_id", "title", "category_name", "rating", "price_gbp", "price_inr"]
    ]
    .sort_values(by=["category_name", "rating"], ascending=[True, False])
    .reset_index(drop=True)
)
print("\nReproduced JOIN result using pd.merge (No SQL)")
display(df_merge_result.head())



Readback 1: Query 1 via pd.read_sql (5-star in-stock books)


,book_id,title,price_gbp,rating
0,11,"1,000 Places to See Before You Die",26.08,5
1,20,A Time of Torment (Charlie Parker #14),48.35,5
2,29,What Happened on Beale Street (Secrets of the ...,25.37,5
3,30,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5
4,34,A Flight of Arrows (The Pathfinders #2),55.53,5
5,36,Mrs. Houdini,30.25,5
6,45,The Passion of Dolssa,28.32,5
7,47,Voyager (Outlander #3),21.07,5
8,48,The Red Tent,35.66,5
9,52,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5



Readback 2: Query 5 (JOIN) via pd.read_sql


,book_id,title,category_name,rating,price_gbp,price_inr
0,34,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,36,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
2,45,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760
3,47,Voyager (Outlander #3),Historical Fiction,5,21.07,2222.885
4,48,The Red Tent,Historical Fiction,5,35.66,3762.130



Reproduced JOIN result using pd.merge (No SQL)


,book_id,title,category_name,rating,price_gbp,price_inr
0,34,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.415
1,36,Mrs. Houdini,Historical Fiction,5,30.25,3191.375
2,45,The Passion of Dolssa,Historical Fiction,5,28.32,2987.760
3,47,Voyager (Outlander #3),Historical Fiction,5,21.07,2222.885
4,48,The Red Tent,Historical Fiction,5,35.66,3762.130
